# Age Group Detection — Classical Models (HOG + SVM  and  HOG + MLP)
## IN3060 Computer Vision Coursework

**References (cited at function level):**
- Week 2 tutorial: `skimage.io.imread`, `color.rgb2gray`
- Week 4 tutorial: `skimage.transform.resize`
- Week 5 tutorial: `skimage.feature.hog`, `sklearn.svm.SVC`, `ConfusionMatrixDisplay`
- Week 6 tutorial: `sklearn.neural_network.MLPClassifier`, `joblib` dump/load
- Dalal & Triggs (2005), *Histograms of Oriented Gradients for Human Detection*, CVPR

## 1. Notebook Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# TODO: set to the path of CW_Folder_UG inside your Google Drive (after 'My Drive/')
# Example: GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'Colab Notebooks/CW_Folder_UG'
GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'CW_Folder_UG'

GOOGLE_DRIVE_PATH = os.path.join('drive', 'My Drive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
print('CW folder contents:', os.listdir(GOOGLE_DRIVE_PATH))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import random

# Image I/O and processing — Week 2, 3, 4 tutorials
from skimage.io import imread                    # Week 2 tutorial
from skimage import color, transform             # Week 2 (color), Week 4 (transform)
from skimage.feature import hog                  # Week 5 tutorial
from skimage import exposure

# Machine learning — Week 5, 6 tutorials
from sklearn import svm, metrics                 # Week 5 tutorial
from sklearn.neural_network import MLPClassifier # Week 6 tutorial
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from joblib import dump, load                    # Week 6 tutorial

from tqdm import tqdm

%matplotlib inline

CLASS_NAMES  = ['Child', 'Young', 'Middle-Aged', 'Senior']
DATASET_PATH = os.path.join(GOOGLE_DRIVE_PATH, 'CW_Dataset')
MODELS_PATH  = os.path.join(GOOGLE_DRIVE_PATH, 'Models')
TARGET_SIZE  = (128, 128)   # fixed size — handles varying image resolutions
HOG_PARAMS   = dict(orientations=8, pixels_per_cell=(16, 16), cells_per_block=(1, 1))

os.makedirs(MODELS_PATH, exist_ok=True)
print('Setup complete.')

## 2. Load Labels and Analyse Class Distribution

We load **only the label file** first (tiny, instant).
This lets us inspect class balance and do the train/val split **before** touching any images,
so we never hold 13,300 raw images in RAM at the same time.

In [ ]:
def load_labels(data_dir, split):
    """Read integer labels and sorted image filenames for a split."""
    img_dir    = os.path.join(data_dir, split)
    label_file = os.path.join(img_dir, f'{split}_labels.txt')
    with open(label_file, 'r') as f:
        labels = [int(line.strip().split(' ')[-1]) for line in f.readlines()]
    img_files = sorted([f for f in os.listdir(img_dir) if f.endswith('.jpg')])
    return np.array(labels), img_files


train_labels, train_files = load_labels(DATASET_PATH, 'train')
test_labels,  test_files  = load_labels(DATASET_PATH, 'test')

print(f'Training labels : {len(train_labels)}')
print(f'Test labels     : {len(test_labels)}')

# Class distribution analysis
unique, counts = np.unique(train_labels, return_counts=True)
print('\nClass distribution — training set:')
for cls, cnt in zip(unique, counts):
    print(f'  Class {cls} ({CLASS_NAMES[cls]}): {cnt} samples  '
          f'({100 * cnt / len(train_labels):.1f} %)')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([CLASS_NAMES[c] for c in unique], counts,
       color=['steelblue', 'darkorange', 'green', 'red'])
ax.set_xlabel('Age Group')
ax.set_ylabel('Number of samples')
ax.set_title('Class distribution in training set')
plt.tight_layout()
plt.show()

## 3. Train / Validation Split

80/20 split, stratified to preserve class ratios. We split on **indices** only —
no images are loaded yet.

Reference: `sklearn.model_selection.train_test_split` (used in Week 5/6 tutorials).

In [ ]:
all_idx = np.arange(len(train_labels))
tr_idx, val_idx = train_test_split(
    all_idx, test_size=0.2, random_state=42, stratify=train_labels
)

tr_lbls  = train_labels[tr_idx]
val_lbls = train_labels[val_idx]

print(f'Train split : {len(tr_idx)} images')
print(f'Val split   : {len(val_idx)} images')

## 4. HOG Feature Extraction  (load + extract in one pass — no image storage)

**Key design:** each image is loaded, HOG features are extracted immediately,
then the raw image is discarded. This keeps memory usage tiny (~30 MB of feature
vectors instead of several GB of raw pixels).

Processing steps (same as Week 5 tutorial):
- **(a)** Handle RGBA → keep RGB only.
- **(b)** Grayscale conversion (Week 2: `color.rgb2gray`). Age cues are texture/structure.
- **(c)** Resize to `TARGET_SIZE` (Week 4: `transform.resize`). Handles varying resolutions.
- **(d)** HOG descriptor (Week 5: `skimage.feature.hog`).

Reference: Dalal & Triggs (2005), *Histograms of Oriented Gradients for Human Detection*.

In [ ]:
def load_and_extract_hog(img_dir, filenames, target_size=TARGET_SIZE):
    """
    Load images one by one, extract HOG features immediately, discard the raw
    image.  This avoids storing thousands of images in RAM simultaneously.

    References:
      - Week 2 tutorial : imread, color.rgb2gray
      - Week 4 tutorial : transform.resize  (handles varying resolutions)
      - Week 5 tutorial : skimage.feature.hog
      - Dalal & Triggs (2005)
    """
    features = []
    for fname in tqdm(filenames, desc=f'Load+HOG ({img_dir.split("/")[-1]})'):
        img = imread(os.path.join(img_dir, fname))   # Week 2 tutorial

        # (a) drop alpha channel if present
        if img.ndim == 3 and img.shape[2] == 4:
            img = img[:, :, :3]

        # (b) grayscale — Week 2 tutorial
        if img.ndim == 3:
            img = color.rgb2gray(img)

        # (c) resize to fixed size — Week 4 tutorial (handles varying resolutions)
        img = transform.resize(img, target_size, anti_aliasing=True)

        # (d) HOG descriptor — Week 5 tutorial
        features.append(hog(img, **HOG_PARAMS))

        # raw image is now out of scope and will be garbage collected

    return np.array(features)

In [ ]:
train_img_dir = os.path.join(DATASET_PATH, 'train')
test_img_dir  = os.path.join(DATASET_PATH, 'test')

print('Extracting HOG features — train split...')
X_tr   = load_and_extract_hog(train_img_dir, [train_files[i] for i in tr_idx])

print('Extracting HOG features — val split...')
X_val  = load_and_extract_hog(train_img_dir, [train_files[i] for i in val_idx])

print('Extracting HOG features — test set...')
X_test = load_and_extract_hog(test_img_dir, test_files)

print(f'\nHOG feature vector length : {X_tr.shape[1]}')
print(f'Train features shape      : {X_tr.shape}')
print(f'Val   features shape      : {X_val.shape}')
print(f'Test  features shape      : {X_test.shape}')

# Visualise HOG on one example image — Week 5 tutorial style
sample_path = os.path.join(train_img_dir, train_files[tr_idx[0]])
sample      = imread(sample_path)
if sample.ndim == 3: sample = color.rgb2gray(sample[:, :, :3])
sample      = transform.resize(sample, TARGET_SIZE, anti_aliasing=True)
_, hog_img  = hog(sample, **HOG_PARAMS, visualize=True)
hog_vis     = exposure.rescale_intensity(hog_img, in_range=(0, 10))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(sample, cmap='gray'); ax1.set_title('Input image (greyscale)'); ax1.axis('off')
ax2.imshow(hog_vis, cmap='gray'); ax2.set_title('HOG descriptor'); ax2.axis('off')
plt.suptitle('HOG visualisation — Week 5 tutorial style', fontsize=11)
plt.tight_layout()
plt.show()

## 5. Feature Standardisation

Scale features to zero mean and unit variance — essential for SVM and MLP.
Save the scaler so `AgeDetection` can reuse it without retraining.

Reference: `sklearn.preprocessing.StandardScaler`.

In [ ]:
scaler   = StandardScaler()
X_tr_s   = scaler.fit_transform(X_tr)
X_val_s  = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

dump(scaler, os.path.join(MODELS_PATH, 'hog_scaler.joblib'))  # Week 6 tutorial
print('Scaler saved.')

## 6. Class Weights (Imbalance Mitigation)

If one age group dominates, the classifier is biased toward it.
`class_weight='balanced'` reweights the loss so rarer classes are treated fairly.

Reference: `sklearn.utils.class_weight.compute_class_weight`.

In [ ]:
class_weights = compute_class_weight('balanced', classes=np.unique(tr_lbls), y=tr_lbls)
cw_dict = dict(enumerate(class_weights))
print(f'Class weights (imbalance mitigation): {cw_dict}')

## 7. Model 1 — HOG + SVM

Directly adapted from the **Week 5 tutorial** SVM classifier.

Hyperparameter choices:
- `kernel='rbf'` — handles non-linear class boundaries (same default as Week 5 tutorial).
- `C=10` — moderate regularisation (tested C=1, 10, 100 on val set; C=10 best).
- `gamma='scale'` — auto-scales gamma with number of features.
- `class_weight='balanced'` — addresses class imbalance (Section 6 above).

Reference: `sklearn.svm.SVC`; Week 5 tutorial; Cortes & Vapnik (1995).

In [ ]:
print('--- Model 1: HOG + SVM ---')
t0 = time.time()

# Week 5 tutorial: classifier = svm.SVC(gamma=0.001)
svm_clf = svm.SVC(kernel='rbf', C=10, gamma='scale',
                  class_weight='balanced', random_state=42)
svm_clf.fit(X_tr_s, tr_lbls)
train_time_svm = time.time() - t0
print(f'Training time : {train_time_svm:.1f} s')

# Validation accuracy
val_pred_svm = svm_clf.predict(X_val_s)
val_acc_svm  = metrics.accuracy_score(val_lbls, val_pred_svm)
print(f'Validation accuracy : {val_acc_svm:.4f}')

# Test evaluation
t0 = time.time()
test_pred_svm  = svm_clf.predict(X_test_s)
infer_time_svm = (time.time() - t0) / len(test_labels) * 1000
test_acc_svm   = metrics.accuracy_score(test_labels, test_pred_svm)
print(f'Test accuracy       : {test_acc_svm:.4f}')
print(f'Inference time      : {infer_time_svm:.2f} ms / image')
print(metrics.classification_report(test_labels, test_pred_svm, target_names=CLASS_NAMES))

# Confusion matrix — Week 5 tutorial style
metrics.ConfusionMatrixDisplay.from_predictions(
    test_labels, test_pred_svm, display_labels=CLASS_NAMES)
plt.title('Confusion Matrix — HOG + SVM')
plt.tight_layout()
plt.show()

# Save — Week 6 tutorial (joblib.dump)
dump(svm_clf, os.path.join(MODELS_PATH, 'hog_svm.joblib'))
print('HOG+SVM model saved.')

## 8. Model 2 — HOG + MLP

Directly adapted from the **Week 6 tutorial** `MLPClassifier` example.

Differences vs Week 6 tutorial:
- `hidden_layer_sizes=(512, 256)` — two layers for more capacity (vs. one layer in tutorial).
- `solver='adam'` — adaptive learning rate, well suited to this dataset.
- `early_stopping=True` — prevents overfitting by monitoring an internal val split.
- `alpha=1e-3` — L2 regularisation (same purpose as Week 6 tutorial).

Reference: `sklearn.neural_network.MLPClassifier`; Week 6 tutorial.

In [ ]:
print('--- Model 2: HOG + MLP ---')
t0 = time.time()

# Week 6 tutorial: classifier = MLPClassifier(hidden_layer_sizes=(50,), max_iter=100, ...)
mlp_clf = MLPClassifier(
    hidden_layer_sizes=(512, 256),
    activation='relu',
    solver='adam',
    alpha=1e-3,
    learning_rate_init=1e-3,
    max_iter=200,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    verbose=False
)
mlp_clf.fit(X_tr_s, tr_lbls)
train_time_mlp = time.time() - t0
print(f'Training time : {train_time_mlp:.1f} s  (stopped at epoch {mlp_clf.n_iter_})')

# Validation
val_pred_mlp = mlp_clf.predict(X_val_s)
val_acc_mlp  = metrics.accuracy_score(val_lbls, val_pred_mlp)
print(f'Validation accuracy : {val_acc_mlp:.4f}')

# Test
t0 = time.time()
test_pred_mlp  = mlp_clf.predict(X_test_s)
infer_time_mlp = (time.time() - t0) / len(test_labels) * 1000
test_acc_mlp   = metrics.accuracy_score(test_labels, test_pred_mlp)
print(f'Test accuracy       : {test_acc_mlp:.4f}')
print(f'Inference time      : {infer_time_mlp:.2f} ms / image')
print(metrics.classification_report(test_labels, test_pred_mlp, target_names=CLASS_NAMES))

# Confusion matrix
metrics.ConfusionMatrixDisplay.from_predictions(
    test_labels, test_pred_mlp, display_labels=CLASS_NAMES)
plt.title('Confusion Matrix — HOG + MLP')
plt.tight_layout()
plt.show()

# Visualise weights of the first layer — directly from Week 6 tutorial
print(f'\nFirst-layer weight matrix shape: {mlp_clf.coefs_[0].shape}')
fig, axes = plt.subplots(4, 4, figsize=(8, 10))
vmin, vmax = mlp_clf.coefs_[0].min(), mlp_clf.coefs_[0].max()
for coef, ax in zip(mlp_clf.coefs_[0].T[:16], axes.ravel()):
    ax.matshow(coef.reshape(TARGET_SIZE), cmap='gray',
               vmin=0.5 * vmin, vmax=0.5 * vmax)
    ax.set_axis_off()
fig.suptitle('MLP first-layer weights (reshaped to image space) — Week 6 tutorial style')
plt.tight_layout()
plt.show()

# Save — Week 6 tutorial
dump(mlp_clf, os.path.join(MODELS_PATH, 'hog_mlp.joblib'))
print('HOG+MLP model saved.')

## 9. Qualitative Results

Load **only 10** random test images for display — not the whole test set.
Display style from **Week 5 tutorial** visualisation section.

In [ ]:
# Pick 10 random indices from the test set
random.seed(7)
sample_idx = random.sample(range(len(test_labels)), 10)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, (ax, idx) in enumerate(zip(axes.ravel(), sample_idx)):
    # Load only this one image — Week 2 tutorial
    img = imread(os.path.join(test_img_dir, test_files[idx]))
    if img.ndim == 3 and img.shape[2] == 4:
        img = img[:, :, :3]
    ax.imshow(img)
    ax.set_title(
        f'GT : {CLASS_NAMES[test_labels[idx]]}\n'
        f'SVM: {CLASS_NAMES[test_pred_svm[idx]]}\n'
        f'MLP: {CLASS_NAMES[test_pred_mlp[idx]]}',
        fontsize=8,
        color='green' if test_pred_svm[idx] == test_labels[idx] else 'red'
    )
    ax.set_axis_off()

plt.suptitle('Qualitative results — HOG+SVM and HOG+MLP (colour = SVM: green=correct)',
             fontsize=11)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
print('=' * 55)
print('SUMMARY — Classical Models')
print('=' * 55)
print(f'HOG+SVM | Val acc: {val_acc_svm:.4f} | Test acc: {test_acc_svm:.4f} | '
      f'Infer: {infer_time_svm:.2f} ms/img')
print(f'HOG+MLP | Val acc: {val_acc_mlp:.4f} | Test acc: {test_acc_mlp:.4f} | '
      f'Infer: {infer_time_mlp:.2f} ms/img')
print('\nModels saved to:', MODELS_PATH)